In [42]:
# PASO 1: Cargar librerías
import pandas as pd

In [43]:
# PASO 2: Leer los archivos
clientes = pd.read_csv("/Users/saraelbachouti/Desktop/UCV-Churn/data/raw/clientes.csv")
facturacion = pd.read_csv("/Users/saraelbachouti/Desktop/UCV-Churn/data/raw/facturacion_mensual.csv")
churn = pd.read_csv("/Users/saraelbachouti/Desktop/UCV-Churn/data/raw/churn_target.csv")

In [44]:
# PASO 3: Convertir las fechas
facturacion["fecha"] = pd.to_datetime(facturacion["fecha"], format="mixed", dayfirst=True)
churn["fecha"] = pd.to_datetime(churn["fecha"], format="mixed", dayfirst=True)

In [45]:
# PASO 4: Juntar los 3 archivos en una sola tabla
df = facturacion.merge(churn, on=["cliente_id", "fecha"], how="inner")
df = df.merge(clientes, on="cliente_id", how="left")

In [46]:
# PASO 5: Ver la tabla
df.head()

,cliente_id,fecha,zona_id_x,tipo_plan_x,num_lineas_x,cargo_base,consumo_extra,descuento_aplicado,importe_total,dias_retraso_pago,...,poblacion_zona,edad,sexo,estado_civil,num_lineas_y,tipo_plan_y,tipo_dispositivo,ingreso_estimado,antiguedad_meses,descuento_activo
0,C000001,2023-01-01,Z26,Prepago,2,43.65,6.98,0.0,50.62,0,...,107159,18.0,M,Soltero/a,2,Prepago,Gama alta,4335.0,72.0,0
1,C000001,2023-01-02,Z26,Prepago,2,43.65,10.33,0.0,53.98,0,...,107159,18.0,M,Soltero/a,2,Prepago,Gama alta,4335.0,72.0,0
2,C000001,2023-01-03,Z26,Prepago,2,43.65,10.87,0.0,54.52,0,...,107159,18.0,M,Soltero/a,2,Prepago,Gama alta,4335.0,72.0,0
3,C000001,2023-01-04,Z26,Prepago,2,43.65,6.82,0.0,50.47,0,...,107159,18.0,M,Soltero/a,2,Prepago,Gama alta,4335.0,72.0,0
4,C000001,2023-01-05,Z26,Prepago,2,43.65,13.26,0.0,56.90,0,...,107159,18.0,M,Soltero/a,2,Prepago,Gama alta,4335.0,72.0,0


In [47]:
# PASO 6: Ver si hay datos vacíos
df.isnull().sum()

cliente_id                  0
fecha                       0
zona_id_x                   0
tipo_plan_x              9788
num_lineas_x                0
cargo_base                  0
consumo_extra               0
descuento_aplicado          0
importe_total            9790
dias_retraso_pago           0
impago_flag                 0
variacion_consumo_pct       0
stress_calidad_lag          0
incidencia_masiva_lag       0
churn                       0
zona_id_y                   0
region                      0
tipo_zona                   0
poblacion_zona              0
edad                     9783
sexo                     1393
estado_civil             2562
num_lineas_y                0
tipo_plan_y                 0
tipo_dispositivo            0
ingreso_estimado         9803
antiguedad_meses         9769
descuento_activo            0
dtype: int64

In [48]:
# PASO 7: Rellenar datos vacíos de forma sencilla
df["sexo"] = df["sexo"].fillna("Desconocido")
df["estado_civil"] = df["estado_civil"].fillna("Desconocido")
df["tipo_plan_x"] = df["tipo_plan_x"].fillna("Desconocido")

df["edad"] = df["edad"].fillna(df["edad"].median())
df["ingreso_estimado"] = df["ingreso_estimado"].fillna(df["ingreso_estimado"].median())
df["antiguedad_meses"] = df["antiguedad_meses"].fillna(df["antiguedad_meses"].median())
df["importe_total"] = (
    df["cargo_base"].fillna(0)
    + df["consumo_extra"].fillna(0)
    - df["descuento_aplicado"].fillna(0)
)

df["importe_total"].isnull().sum()

np.int64(0)

In [49]:
df = df.drop_duplicates()

In [50]:
# PASO 8: Ordenar por cliente y fecha
df = df.sort_values(["cliente_id", "fecha"])

In [51]:
# PASO 9: Crear variables fáciles de entender

# Factura del mes anterior
df["factura_mes_anterior"] = df.groupby("cliente_id")["importe_total"].shift(1)

# Cambio de factura respecto al mes anterior
df["subida_factura"] = df["importe_total"] - df["factura_mes_anterior"]

# Consumo del mes anterior
df["consumo_mes_anterior"] = df.groupby("cliente_id")["consumo_extra"].shift(1)

# Cambio de consumo
df["cambio_consumo"] = df["consumo_extra"] - df["consumo_mes_anterior"]

# Impago del mes anterior
df["impago_mes_anterior"] = df.groupby("cliente_id")["impago_flag"].shift(1)

# Retraso de pago del mes anterior
df["retraso_mes_anterior"] = df.groupby("cliente_id")["dias_retraso_pago"].shift(1)

In [52]:
print(df.isnull().sum()[df.isnull().sum() > 0])

factura_mes_anterior    10000
subida_factura          10000
consumo_mes_anterior    10000
cambio_consumo          10000
impago_mes_anterior     10000
retraso_mes_anterior    10000
dtype: int64


In [53]:
columnas_lag = [
    "factura_mes_anterior",
    "consumo_mes_anterior",
    "retraso_mes_anterior",
    "impago_mes_anterior",
    "cambio_consumo",
    "subida_factura"
]

df[columnas_lag] = df[columnas_lag].fillna(0)

In [54]:
# PASO 10: Ver si los impagos tienen relación con el churn
df.groupby("impago_flag")["churn"].mean()

impago_flag
0    0.005117
1    0.015626
Name: churn, dtype: float64

In [55]:
# PASO 11: Ver si los retrasos de pago influyen
df.groupby("retraso_mes_anterior")["churn"].mean()

retraso_mes_anterior
0.0       0.004975
5.0       0.011518
6.0       0.017467
7.0       0.000000
8.0       0.024096
            ...   
940.0     0.000000
960.0     0.000000
980.0     0.000000
1000.0    0.000000
1100.0    0.000000
Name: churn, Length: 106, dtype: float64

In [56]:
# PASO 12: Ver correlaciones simples
df[[
    "churn",
    "importe_total",
    "subida_factura",
    "consumo_extra",
    "cambio_consumo",
    "impago_flag",
    "impago_mes_anterior",
    "dias_retraso_pago",
    "retraso_mes_anterior"
]].corr()["churn"].sort_values(ascending=False)

churn                   1.000000
impago_mes_anterior     0.047021
impago_flag             0.040665
retraso_mes_anterior    0.021364
dias_retraso_pago       0.020607
subida_factura          0.000663
cambio_consumo         -0.001504
consumo_extra          -0.006831
importe_total          -0.012877
Name: churn, dtype: float64

In [57]:
# PASO 13: Crear una regla sencilla de alerta

df["alerta_churn"] = 0

df.loc[
    (df["impago_flag"] == 1) |
    (df["impago_mes_anterior"] == 1) |
    (df["dias_retraso_pago"] > 15),
    "alerta_churn"
] = 1

In [58]:
# PASO 14: Ver si la alerta funciona
df.groupby("alerta_churn")["churn"].mean()

alerta_churn
0    0.004128
1    0.014940
Name: churn, dtype: float64

Hipótesis posibles del análisis
Hipótesis 1
Los clientes con impagos presentan una mayor probabilidad de churn.
Hipótesis 2
Un aumento en los días de retraso de pago está relacionado con una mayor tasa de abandono de clientes.
Hipótesis 3
Los clientes que experimentan incidencias de calidad o problemas de red tienen más probabilidad de abandonar el servicio.
Hipótesis 4
Las variaciones negativas en el consumo pueden actuar como una señal temprana de churn.
Hipótesis 5
Los clientes con menor antigüedad tienden a abandonar más fácilmente que los clientes con más tiempo en la compañía.
Hipótesis 6
Los cambios bruscos en la factura mensual pueden influir en la decisión de abandono de los clientes.
Hipótesis 7
Los clientes con alertas de riesgo (alerta_churn = 1) presentan una tasa de churn superior al resto de usuarios.


Tras realizar el proceso de limpieza y análisis exploratorio de los datos, se observó que variables relacionadas con impagos, retrasos de pago y calidad del servicio presentan relación con el churn de clientes. Además, algunas variables temporales, como los datos del mes anterior y las variaciones de consumo o facturación, permiten identificar patrones previos al abandono.
Durante el proceso de preparación de datos se detectaron valores nulos y registros duplicados, los cuales fueron tratados adecuadamente para garantizar la calidad del dataset final. Una vez completada la limpieza, el conjunto de datos quedó preparado para futuros modelos predictivos de Machine Learning orientados a la detección temprana de clientes con riesgo de churn.
En general, el análisis sugiere que los problemas financieros y operativos pueden influir directamente en la pérdida de clientes, por lo que la empresa podría utilizar este tipo de modelos para anticiparse al abandono y mejorar la retención de usuarios.